In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-notifications")

ruta_driver = r"D:\Users\Lenovo\Documents\chrome-win\chromedriver.exe" 
service = Service(ruta_driver)
driver = webdriver.Chrome(service=service, options=chrome_options)

# ============================================================================
# CREDENCIALES Y VARIABLES
# ============================================================================

URL = "http://senasofiaplus.edu.co/sofia-public/"
USUARIO = "1050962935" 
CONTRASENA = "PapaJose92805331050*" 
FICHA_A_INGRESAR = "123456"

# IDs de iframes
IFRAME_LOGIN = 'registradoBox1'
IFRAME_CONTENIDO = 'contenido'
IFRAME_MODAL = 'modalDialogContentviewDialog'  # ← ¡Nota la 'v' minúscula!

# XPaths
XPATH_ICONO_FILTROS = '/html/body[1]/div[2]/div[1]/fieldset/form/table/tbody/tr/td[3]/a/img'
XPATH_INPUT_FICHA = '/html/body/div[2]/form/fieldset/div/table/tbody/tr[1]/td[2]/input'
XPATH_BOTON_BUSCAR = '//*[@id="form:buscarCBT"]'

# ============================================================================
# FUNCIONES
# ============================================================================

def imprimir_seccion(titulo):
    """Imprime un título de sección formateado"""
    print("\n" + "="*70)
    print(f"  {titulo}")
    print("="*70)

def login():
    """Realiza el inicio de sesión"""
    imprimir_seccion("🔐 INICIANDO SESIÓN")
    
    try:
        # Clic en botón "Ingresar"
        try:
            boton_ingresar = driver.find_element(By.XPATH, "//a[contains(text(), 'Ingresar')]")
            boton_ingresar.click()
            time.sleep(2)
        except Exception:
            pass
        
        # Cambiar al iframe de login
        driver.switch_to.default_content()
        driver.switch_to.frame(IFRAME_LOGIN)
        
        # Ingresar credenciales
        input_usuario = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[2]/input"))
        )
        input_usuario.send_keys(USUARIO)
        
        input_contrasena = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[3]/input"))
        )
        input_contrasena.send_keys(CONTRASENA)
        
        # Hacer clic en botón de login
        boton_login = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "/html/body/form/div/div/div/div[7]/input"))
        )
        boton_login.click()
        time.sleep(5)
        
        print("✅ Inicio de sesión exitoso")
        return True
        
    except Exception as e:
        print(f"❌ Error durante el login: {e}")
        return False

def navegar_a_convocar_aspirantes():
    """Navega hasta la sección de Convocar Aspirantes"""
    imprimir_seccion("🧭 NAVEGANDO A 'CONVOCAR ASPIRANTES'")
    
    try:
        # Seleccionar rol (opción 4)
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="seleccionRol:roles"]/option[4]'))
        ).click()
        time.sleep(4)
        
        # Navegar por el menú lateral
        driver.switch_to.default_content()
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="side-menu"]/li[5]/a'))
        ).click()
        
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="side-menu"]/li[5]/ul/li[2]/a'))
        ).click()
        
        # Clic en "Convocar aspirantes"
        elemento_convocar = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="185Opcion"]'))
        )
        driver.execute_script("arguments[0].click();", elemento_convocar)
        time.sleep(3)
        
        print("✅ Navegación exitosa a 'Convocar aspirantes'")
        return True
        
    except Exception as e:
        print(f"❌ Error durante la navegación: {e}")
        return False

def abrir_modal_filtros():
    """Abre el modal de filtros haciendo clic en el ícono"""
    imprimir_seccion("🔍 ABRIENDO MODAL DE FILTROS")
    
    try:
        # Cambiar al iframe principal
        driver.switch_to.default_content()
        WebDriverWait(driver, 20).until(
            EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_CONTENIDO))
        )
        print(f"✅ Foco en iframe principal: '{IFRAME_CONTENIDO}'")
        
        # Hacer clic en el ícono de filtros
        icono_filtros = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, XPATH_ICONO_FILTROS))
        )
        driver.execute_script("arguments[0].click();", icono_filtros)
        print("✅ Clic en ícono de filtros ejecutado")
        
        # Esperar a que el modal cargue completamente
        time.sleep(8)
        return True
        
    except Exception as e:
        print(f"❌ Error al abrir modal de filtros: {e}")
        return False

def ingresar_ficha_y_buscar(numero_ficha):
    """Ingresa el número de ficha en el modal y ejecuta la búsqueda"""
    imprimir_seccion(f"📝 INGRESANDO FICHA: {numero_ficha}")
    
    try:
        # CLAVE: Navegar al iframe modal ANIDADO dentro del iframe contenido
        # NO volver a default_content(), ya estamos dentro de 'contenido'
        
        WebDriverWait(driver, 10).until(
            EC.frame_to_be_available_and_switch_to_it((By.ID, IFRAME_MODAL))
        )
        print(f"✅ Foco en iframe modal anidado: '{IFRAME_MODAL}'")
        
        # Encontrar y rellenar el input
        input_ficha = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, XPATH_INPUT_FICHA))
        )
        input_ficha.clear()
        input_ficha.send_keys(numero_ficha)
        print(f"✅ Ficha '{numero_ficha}' ingresada correctamente")
        
        # Hacer clic en el botón "Buscar"
        boton_buscar = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, XPATH_BOTON_BUSCAR))
        )
        boton_buscar.click()
        print("✅ Búsqueda ejecutada con éxito")
        
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"❌ Error al ingresar ficha o ejecutar búsqueda: {e}")
        return False

# ============================================================================
# FLUJO PRINCIPAL
# ============================================================================

def main():
    """Función principal que ejecuta todo el flujo"""
    imprimir_seccion("🚀 INICIANDO AUTOMATIZACIÓN SENA SOFIA PLUS")
    
    try:
        # Abrir la página
        driver.get(URL)
        time.sleep(3)
        print(f"✅ Página cargada: {URL}")
        
        # Ejecutar pasos
        if not login():
            raise Exception("Error en login")
        
        if not navegar_a_convocar_aspirantes():
            raise Exception("Error en navegación")
        
        if not abrir_modal_filtros():
            raise Exception("Error al abrir modal")
        
        if not ingresar_ficha_y_buscar(FICHA_A_INGRESAR):
            raise Exception("Error al buscar ficha")
        
        # Éxito total
        imprimir_seccion("✅✅✅ PROCESO COMPLETADO CON ÉXITO ✅✅✅")
        print(f"   Ficha '{FICHA_A_INGRESAR}' procesada correctamente")
        
        # Mantener navegador abierto para verificación
        print("\n⏸️ Navegador permanecerá abierto por 10 segundos...")
        time.sleep(10)
        
    except Exception as e:
        imprimir_seccion("❌ ERROR CRÍTICO")
        print(f"   {str(e)}")
        print("\n⏸️ Navegador permanecerá abierto por 15 segundos para inspección...")
        time.sleep(15)
        
    finally:
        driver.quit()
        print("\n🔚 Navegador cerrado. Proceso finalizado.")

# ============================================================================
# EJECUCIÓN
# ============================================================================

if __name__ == "__main__":
    main()


  🚀 INICIANDO AUTOMATIZACIÓN SENA SOFIA PLUS
✅ Página cargada: http://senasofiaplus.edu.co/sofia-public/

  🔐 INICIANDO SESIÓN
✅ Inicio de sesión exitoso

  🧭 NAVEGANDO A 'CONVOCAR ASPIRANTES'
❌ Error durante la navegación: Message: element click intercepted: Element <a href="javascript:void(0)">...</a> is not clickable at point (150, 292). Other element would receive the click: <div class="blockUI blockOverlay" style="z-index: 2000; border: none; margin: 0px; padding: 0px; width: 100%; height: 100%; top: 0px; left: 0px; background-color: rgb(0, 0, 0); opacity: 0.6; cursor: wait; position: fixed;"></div>
  (Session info: chrome=103.0.5046.0)
Stacktrace:
Backtrace:
	Ordinal0 [0x01096903+2844931]
	Ordinal0 [0x00F74271+1655409]
	Ordinal0 [0x00E0013A+131386]
	Ordinal0 [0x00E2D1A1+315809]
	Ordinal0 [0x00E2BA4D+309837]
	Ordinal0 [0x00E29F09+302857]
	Ordinal0 [0x00E2946A+300138]
	Ordinal0 [0x00E20BB0+265136]
	Ordinal0 [0x00E3C56C+378220]
	Ordinal0 [0x00E205F1+263665]
	Ordinal0 [0x00E3C894+37